# Basic Question & Answer Generation from a FileSet

This notebook demonstrates the simplest way to generate question-answer pairs from a FileSet. Documents are chunked into seeds, then questions and labels are generated in a single step using `QuestionAndLabelGenerator`.

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [ ]:
%pip install lightningrod-ai python-dotenv -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [ ]:
fileset_id = "PASTE_YOUR_FILESET_ID_HERE"

## Configure the Question Pipeline

- **`FileSetSeedGenerator`** chunks the documents in your FileSet into seeds (text passages)
- **`QuestionAndLabelGenerator`** generates questions and answers in a single step — the simplest way to produce labeled Q&A pairs

In [7]:
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    QuestionAndLabelGenerator,
    FreeResponseAnswerType,
)

answer_type = FreeResponseAnswerType()

pipeline = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionAndLabelGenerator(
        questions_per_seed=2,
        answer_type=answer_type,
        instructions=(
            "Generate questions about the financial metrics, business events, "
            "and forward guidance in these quarterly investor reports. Questions should be "
            "specific and verifiable from the report content."
        ),
    ),
)

## Run the Pipeline

In [8]:
dataset = lr.transforms.run(
    pipeline,
    name="FileSet - Basic QA",
)
print(f"Dataset: {dataset.id}")
print(f"Rows: {dataset.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           4d6156a1-6165-4098-ab16-216aa056e564                                                       │
│                                                                                                                 │
│    Total cost: $0.01                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step                                  ┃ Progress               ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩  │
│  │ FileSetSeedGeneratorTransform         │ Complete               │  10 │  10 │        0 │      0 │       2s │  │
│  │ QuestionAndLabelGeneratorTransform    │ Complete               │  10 │  20 │        0 │      0 │       5s │  │
│  └───────────────────────────────────────┴────────────────────────┴─────┴─────┴──────────┴────────┴──────────┘  │
│                                                                                                                 │
│    View full details:                                                                                           │
│  ]8;id=625720;https://dashboard.lightningrod.ai/?redirect=/datasets/27971a0d-2195-4e93-9121-69cb59d47b96\https://dashboard.lightningrod.ai/?redirect=/datasets/27971a0d-2195-4e93-9121-69cb59d47b96]8;;\                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dataset: 27971a0d-2195-4e93-9121-69cb59d47b96
Rows: 20


> **Note:** This can take a few minutes to complete processing.

## View the Results

In [6]:
%pip install pandas -q

from IPython.display import clear_output
clear_output()

In [9]:
import pandas as pd

samples = dataset.download()
rows = dataset.flattened()
df = pd.DataFrame(rows)

print(f"Generated {dataset.num_rows} samples ({dataset.valid_count() / dataset.num_rows * 100:.1f}% valid)\n")

cols = ["question_text", "label", "label_confidence", "is_valid"]
df[[c for c in cols if c in df.columns]]

Generated 20 samples (100.0% valid)



,question_text,label,label_confidence,is_valid
0,What was the total acquisition cost for CyberS...,"The acquisition closed on February 14, 2025, f...",1.0,True
1,When is the $400M data center expansion in Nor...,It is on schedule for Q4 completion.,1.0,True
2,What is Vanguard Industries' projected revenue...,$7.0B-$7.4B (13-20% growth),1.0,True
3,What are APEX Technologies Inc.'s projected re...,Q4 2024 revenue is expected to be between $2.5...,1.0,True
4,By what percentage did the Robotics-as-a-Servi...,45%,1.0,True
5,What is the expected operating margin range fo...,"16-17%, due to a $400 million investment in da...",1.0,True
6,What is the projected annual run-rate for the ...,$2B+,1.0,True
7,When does APEX Technologies expect AXCompute 4...,General availability for AXCompute 4.0 is targ...,1.0,True
8,What is the specific annual cost reduction tar...,$120 million,1.0,True
9,What percentage of total revenue does manageme...,35-38%,1.0,True
